[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aecelaya/MIST/blob/main/examples/mist_compile_benchmark.ipynb)

# Measuring `torch.compile` in MIST

MIST can compile its networks during training (`training.compile`), but it's off by
default until its effect is known. This notebook measures that effect on each network:
speed, memory, and how long the compile takes to pay for itself.

**Run it once per GPU.** Choose the GPU under *Runtime > Change runtime type*, then
*Runtime > Run all*. It saves a CSV named after the GPU and downloads it. No data is
needed; inputs are random.

Every measurement runs in its own process with an empty compile cache, so one run's
cuDNN tuning or compiled code can't flatter the next. For each network, precision,
and patch size, the same steps run with and without compiling:

- **Training:** real steps with MIST's default loss, AdamW, and gradient clipping,
  starting with the step that does the compiling.
- **Validation:** forward passes without gradients, on the same model, including a
  change of batch size, which can trigger a recompile.

What it leaves out: data loading, which takes the same time either way. Real runs that
wait on data will gain less than the training-step speedup shown here. Multi-GPU
training also isn't measured.

A run that gets disconnected only loses its remaining rows, since results are saved
after every measurement.

## Setup

In [ ]:
!pip install -q "git+https://github.com/aecelaya/MIST.git"

In [ ]:
import json
import os
import platform
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import pandas as pd
import torch

from mist.training import patch_planning as pp
from mist.utils import hardware

warnings.filterwarnings("ignore")

GIB = 1024**3

# Set MIST_COMPILE_DRY_RUN to test this notebook's code without a GPU. Memory then
# reads as 0 and CPU timings say nothing about a GPU.
DRY_RUN = bool(os.environ.get("MIST_COMPILE_DRY_RUN"))
if not (torch.cuda.is_available() or DRY_RUN):
    raise RuntimeError("No GPU found. Choose one under Runtime > Change runtime type.")

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEMORY = float(torch.cuda.get_device_properties(0).total_memory)
else:
    GPU_NAME, GPU_MEMORY = "dry-run", 4.0 * GIB

PRECISIONS = ["fp32"] + (["bf16"] if hardware.bf16_supported() else [])
OUTPUT_CSV = f"mist_compile_{GPU_NAME.replace(' ', '_').replace('/', '-')}.csv"

print(f"GPU: {GPU_NAME}, {GPU_MEMORY / GIB:.1f} GiB")
print(f"PyTorch {torch.__version__}, Python {platform.python_version()}")
print(f"Precisions to measure: {PRECISIONS}")
print(f"Results go to {OUTPUT_CSV}")

## What to measure

One network per family, plus the heavier variants. The patch sizes come from MIST's
planner, for half of this GPU's memory and for all of it. The second is what real
runs use, but it may run out of memory for some networks; the first still gives their
timing.

In [ ]:
NETWORKS = [
    "nnunet-plain",
    "nnunet-res",
    "nnunet-pocket",
    "resenc",
    "fmgnet",
    "wnet",
    "mednext-small",
    "mednext-large",
    "swinunetr-small",
    "primus-medium",
    "primusv2-small",
]
SPACING = [1.0, 1.0, 1.0]
MEMORY_FRACTIONS = [0.5, 1.0]
BATCH_SIZE = 2  # What the planner chooses for large images.
SETTLING_STEPS = 5  # Not timed, after the first step.
TRAINING_STEPS = 40  # Timed, within the limit below.
TRAINING_TIME_LIMIT_S = 180
EVAL_BATCH = 4
EVAL_STEPS = 10
RUN_TIMEOUT_S = 1800  # Gives up on a run that hangs, and records that.
IN_CHANNELS = 4  # As in BraTS.
OUT_CHANNELS = 5
MEDIAN_SIZE = [512, 512, 512]

# For judging whether compiling pays off: MIST's defaults are 250 steps per epoch.
STEPS_PER_EPOCH = 250

if DRY_RUN:
    NETWORKS = ["nnunet-pocket"]
    MEMORY_FRACTIONS = [0.05]
    SETTLING_STEPS = 1
    TRAINING_STEPS = 3
    EVAL_BATCH = 2
    EVAL_STEPS = 1
    MEDIAN_SIZE = [64, 64, 64]

## Measuring

This cell writes the script that runs each measurement in its own process.

In [ ]:
WORKER = r'''"""Measure one network, compiled or not, in its own process."""

import json
import statistics
import sys
import time
import warnings

import torch

from mist.loss_functions.deep_supervision_wrapper import DeepSupervisionLoss
from mist.loss_functions.loss_registry import get_loss
from mist.models.model_registry import get_model_from_registry
from mist.utils import hardware

GIB = 1024**3


def sync():
    """Wait for the GPU, so that timings cover the work."""
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def reset_peak():
    """Free cached memory, so that later peaks leave out what came before."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def peak_gib():
    """Return peak allocated and reserved memory since the last reset."""
    if not torch.cuda.is_available():
        return 0.0, 0.0
    return torch.cuda.max_memory_allocated() / GIB, torch.cuda.max_memory_reserved() / GIB


def timed(function):
    """Return a function's seconds and its result."""
    sync()
    start = time.perf_counter()
    result = function()
    sync()
    return time.perf_counter() - start, result


def measure(job):
    """Run training and validation-style steps on one network."""
    warnings.filterwarnings("ignore")
    hardware.tune_gpu_math()  # As in MIST's training.
    torch.manual_seed(0)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    patch = job["patch"]
    channels = job["in_channels"]

    model = get_model_from_registry(
        job["network"],
        in_channels=channels,
        out_channels=job["out_channels"],
        patch_size=patch,
        target_spacing=job["spacing"],
    ).to(device)
    if job["compiled"]:
        model.compile()  # What training.compile does.
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = DeepSupervisionLoss(get_loss("dice_ce")())

    # One fixed batch, so that making data isn't timed and both runs see the same.
    batch = job["batch"]
    image = torch.randn(batch, channels, *patch, device=device)
    label = torch.randint(0, job["out_channels"], (batch, 1, *patch), device=device)

    def train_step():
        optimizer.zero_grad()
        with hardware.autocast_context(job["amp"]):
            output = model(image)
            loss = criterion(
                y_true=label,
                y_pred=output["prediction"],
                y_supervision=output.get("deep_supervision"),
            )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        return float(loss.detach())

    def predict(size):
        inputs = torch.randn(size, channels, *patch, device=device)

        def run():
            with torch.inference_mode(), hardware.autocast_context(job["amp"]):
                model(inputs)

        return run

    result = {}

    # Training. The first step includes compiling; the next few settle cuDNN and memory.
    model.train()
    result["first_train_step_s"], _ = timed(train_step)
    result["first_step_reserved_gib"] = peak_gib()[1]
    reset_peak()
    for _ in range(job["settling_steps"]):
        loss = train_step()
    steps = []
    limit = time.perf_counter() + job["time_limit_s"]
    for _ in range(job["training_steps"]):
        seconds, loss = timed(train_step)
        steps.append(seconds)
        if time.perf_counter() > limit and len(steps) >= 5:
            break
    result["train_steps_timed"] = len(steps)
    result["train_s_per_step"] = statistics.median(steps)
    result["train_s_per_step_mean"] = statistics.fmean(steps)
    result["final_loss"] = loss
    result["steady_allocated_gib"], result["steady_reserved_gib"] = peak_gib()

    # Validation runs on the same model, with shapes that change as a case runs out.
    model.eval()
    reset_peak()
    result["first_eval_step_s"], _ = timed(predict(job["eval_batch"]))
    steps = [timed(predict(job["eval_batch"]))[0] for _ in range(job["eval_steps"])]
    result["eval_s_per_step"] = statistics.median(steps)
    result["eval_reserved_gib"] = peak_gib()[1]
    result["new_eval_shape_s"], _ = timed(predict(1))

    # Back to training, as after each validation.
    model.train()
    result["back_to_train_step_s"], _ = timed(train_step)
    return result


def main():
    """Print one measurement as JSON, whatever happens."""
    job = json.loads(sys.argv[1])
    try:
        result = {"status": "ok", **measure(job)}
    except torch.cuda.OutOfMemoryError:
        result = {"status": "out of memory"}
    except Exception as error:  # pylint: disable=broad-except
        result = {"status": "error", "error": f"{type(error).__name__}: {error}"[:500]}
    print("RESULT " + json.dumps(result))


if __name__ == "__main__":
    main()
'''
Path("compile_worker.py").write_text(WORKER)
print("Wrote compile_worker.py")

In [ ]:
rows = []


def run_job(job):
    """Run one measurement in a new process with an empty compile cache."""
    with tempfile.TemporaryDirectory() as cache:
        env = dict(os.environ, TORCHINDUCTOR_CACHE_DIR=cache)
        try:
            done = subprocess.run(
                [sys.executable, "compile_worker.py", json.dumps(job)],
                capture_output=True,
                text=True,
                timeout=RUN_TIMEOUT_S,
                env=env,
                check=False,
            )
        except subprocess.TimeoutExpired:
            return {"status": "timed out"}
    for line in reversed(done.stdout.splitlines()):
        if line.startswith("RESULT "):
            return json.loads(line[len("RESULT ") :])
    return {"status": "error", "error": done.stderr[-500:]}


def record(name, precision, patch, estimate, compiled):
    """Run one measurement and save it."""
    job = {
        "network": name,
        "patch": patch,
        "spacing": SPACING,
        "batch": BATCH_SIZE,
        "amp": precision == "bf16",
        "compiled": compiled,
        "in_channels": IN_CHANNELS,
        "out_channels": OUT_CHANNELS,
        "settling_steps": SETTLING_STEPS,
        "training_steps": TRAINING_STEPS,
        "time_limit_s": TRAINING_TIME_LIMIT_S,
        "eval_batch": EVAL_BATCH,
        "eval_steps": EVAL_STEPS,
    }
    row = {
        "gpu": GPU_NAME,
        "gpu_memory_gib": round(GPU_MEMORY / GIB, 2),
        "torch": torch.__version__,
        "network": name,
        "patch": "x".join(str(p) for p in patch),
        "batch": BATCH_SIZE,
        "precision": precision,
        "compiled": compiled,
        "planner_estimate_gib": pp.estimated_bytes(estimate, BATCH_SIZE, precision == "bf16") / GIB,
        **run_job(job),
    }
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
    label = "compiled" if compiled else "plain   "
    if row["status"] == "ok":
        print(
            f"{name:16} {row['patch']:14} {precision} {label}: "
            f"first step {row['first_train_step_s']:6.1f} s, "
            f"then {row['train_s_per_step']:.3f} s, {row['steady_reserved_gib']:5.1f} GiB"
        )
    else:
        print(f"{name:16} {row['patch']:14} {precision} {label}: {row['status']}")
        if row.get("error"):
            print(f"    {row['error']}")


for name in NETWORKS:
    for precision in PRECISIONS:
        patches = []
        for fraction in MEMORY_FRACTIONS:
            plan = pp.plan_patch_and_batch(
                architecture=name,
                in_channels=IN_CHANNELS,
                out_channels=OUT_CHANNELS,
                median_size=MEDIAN_SIZE,
                target_spacing=SPACING,
                gpu_memory_bytes=GPU_MEMORY * fraction,
                amp=precision == "bf16",
                num_cases=10000,
                batch_size=BATCH_SIZE,
            )
            if plan.patch_size not in patches:
                patches.append(plan.patch_size)
        for patch in patches:
            estimate = pp.estimate_network(name, IN_CHANNELS, OUT_CHANNELS, patch, SPACING)
            for compiled in (False, True):
                record(name, precision, patch, estimate, compiled)

results = pd.DataFrame(rows)
print(f"\nDone: {len(results)} measurements.")
print(results["status"].value_counts().to_string())

## Results

Above 1 under *speedup* means compiling made steps faster, and above 1 under
*memory* means it used more. *Break-even* is how many training steps the time saved
takes to repay the compile. A negative or missing value means it never does.
*Loss gap* compares the loss after the same steps with and without compiling; it
should be small, since both train on the same fixed batch from the same weights.

In [ ]:
KEYS = ["network", "precision", "patch"]
ok = results[results["status"] == "ok"]
plain = ok[~ok["compiled"]].set_index(KEYS)
compiled = ok[ok["compiled"]].set_index(KEYS)
both = plain.index.intersection(compiled.index)

if len(both):
    plain, compiled = plain.loc[both], compiled.loc[both]
    saved = plain["train_s_per_step"] - compiled["train_s_per_step"]
    cost = compiled["first_train_step_s"] - plain["first_train_step_s"]
    summary = pd.DataFrame(
        {
            "plain_s": plain["train_s_per_step"],
            "compiled_s": compiled["train_s_per_step"],
            "train_speedup": plain["train_s_per_step"] / compiled["train_s_per_step"],
            "eval_speedup": plain["eval_s_per_step"] / compiled["eval_s_per_step"],
            "compile_cost_s": cost,
            "break_even_steps": (cost / saved).where(saved > 0),
            "new_eval_shape_s": compiled["new_eval_shape_s"],
            "train_memory": compiled["steady_reserved_gib"]
            / plain["steady_reserved_gib"].clip(lower=1e-9),
            "eval_memory": compiled["eval_reserved_gib"]
            / plain["eval_reserved_gib"].clip(lower=1e-9),
            "loss_gap": (compiled["final_loss"] - plain["final_loss"]).abs(),
        }
    )
    print(summary.round(3).to_string())
    print(f"\nFor scale, an epoch at MIST's defaults is {STEPS_PER_EPOCH} steps.")
else:
    print("No network finished both with and without compiling.")

failed = results[results["status"] != "ok"]
if len(failed):
    columns = ["network", "precision", "patch", "compiled", "status"]
    print("\nDidn't finish:")
    print(failed[columns].to_string(index=False))

## Save the results

Downloads the CSV. If the download doesn't start, it's in the Files panel on the
left.

In [ ]:
print(f"Saved {OUTPUT_CSV}")
try:
    from google.colab import files

    files.download(OUTPUT_CSV)
except ImportError:
    pass